# Code Writer → Reviewer → Recommender → Doc Agent

Demonstrates a mini pipeline with four agents (writer, reviewer, recommender, docs) using a stubbed LLM or OpenAI.

In [1]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY (press Enter to skip): ")

from agentic_codex import AgentBuilder, Context, FunctionAdapter, EnvOpenAIAdapter
from agentic_codex.core.schemas import AgentStep, Message

def stub_llm(prompt: str) -> str:
    lines = [ln.strip() for ln in prompt.splitlines() if ln.strip()]
    return f"[stub llm] {lines[-1][:160] if lines else 'response'}"

try:
    llm = EnvOpenAIAdapter(model="gpt-4o-mini")
except Exception:
    llm = FunctionAdapter(stub_llm)

def make_agent(name: str, role: str, template: str):
    def step(ctx: Context) -> AgentStep:
        content = template.format(goal=ctx.goal, code=ctx.scratch.get("code", ""), review=ctx.scratch.get("review", ""))
        return AgentStep(out_messages=[Message(role=name, content=content)], state_updates={role: content})
    return AgentBuilder(name=name, role=role).with_llm(llm).with_step(step).build()

writer = make_agent("writer", "code", "def solution():\n    return 'hello'  # goal: {goal}")
reviewer = make_agent("reviewer", "review", "Review for goal '{goal}': ensure code returns hello string. Found: {code}")
recommender = make_agent("recommender", "recommendation", "Improve goal '{goal}': add docstring, tests; current: {code}")
doc_agent = make_agent("doc_agent", "docs", "Documentation for goal '{goal}': summarize code and recommendations.")

ctx = Context(goal="Return hello", scratch={})

for agent in (writer, reviewer, recommender, doc_agent):
    result = agent.run(ctx)
    if result.out_messages:
        ctx.push_message(result.out_messages[-1])
    ctx.scratch.update(result.state_updates)

ctx.scratch

{'_metrics': {'counters': {},
  'latency': {('agent.latency',
    (('agent', 'writer'),
     ('role',
      'code'))): {'count': 1, 'total': 0.000421667005866766, 'min': 0.000421667005866766, 'max': 0.000421667005866766},
   ('agent.latency', (('agent', 'reviewer'), ('role', 'review'))): {'count': 1,
    'total': 1.091597368940711e-05,
    'min': 1.091597368940711e-05,
    'max': 1.091597368940711e-05},
   ('agent.latency',
    (('agent', 'recommender'),
     ('role',
      'recommendation'))): {'count': 1, 'total': 7.0410314947366714e-06, 'min': 7.0410314947366714e-06, 'max': 7.0410314947366714e-06},
   ('agent.latency', (('agent', 'doc_agent'), ('role', 'docs'))): {'count': 1,
    'total': 5.209003575146198e-06,
    'min': 5.209003575146198e-06,
    'max': 5.209003575146198e-06}}},
 'code': "def solution():\n    return 'hello'  # goal: Return hello",
 'review': "Review for goal 'Return hello': ensure code returns hello string. Found: def solution():\n    return 'hello'  # goal: Retur